# Lahore Night Lights (VIIRS monthly) — Oct 2024

In [1]:
# ==========================================================
# Outputs: GeoTIFF (radiance), point grid (GeoJSON + CSV), UC stats
# ==========================================================
import os, ee, geemap, geopandas as gpd, pandas as pd

# ---------- 0) EE init ----------
try:
    ee.Initialize()
except Exception:
    ee.Authenticate(auth_mode='localhost')
    ee.Initialize()


/Users/ahmed/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:


# --- region (reuse yours) ---
UC_SHP = "../../data/Lahore UCs/Lahore UC.shp"
gdf = gpd.read_file(UC_SHP).to_crs(4326)
gdf["geometry"] = gdf["geometry"].buffer(0)
ucs_fc = geemap.gdf_to_ee(gdf)
region = ucs_fc.geometry()

# --- VIIRS collection picker (same as before) ---
DATASETS = [
    "NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG",
    "NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG",
    "NOAA/VIIRS/DNB/MONTHLY_V2/VCMSLCFG",
    "NOAA/VIIRS/DNB/MONTHLY_V21/VCMSLCFG",
]
def get_viirs_monthly():
    for ds in DATASETS:
        try:
            _ = ee.ImageCollection(ds).limit(1).size().getInfo()
            return ds
        except Exception:
            continue
    raise RuntimeError("No VIIRS monthly collection found.")
VIIRS = get_viirs_monthly()
print("[OK] Using:", VIIRS)

START, END = "2024-10-01", "2024-10-31"
SAMPLE_SCALE = 200
SMOOTH_RADIUS_M = 0  # e.g., 500 to soften

# --- FIXED: server-side-safe map function ---
def prep_viirs(img):
    rad = img.select("avg_rad")
    rad = rad.updateMask(rad.gte(0))
    bandNames = img.bandNames()                           # ee.List
    hasCvg = bandNames.contains("cf_cvg")                 # ee.ComputedObject (Bool)
    # if has cf_cvg: mask by cf_cvg>0 else mask=1
    cvg_mask = ee.Image(ee.Algorithms.If(hasCvg,
                                         img.select("cf_cvg").gt(0),
                                         ee.Image(1)))
    rad = rad.updateMask(cvg_mask)
    return rad.rename("rad")

col = (ee.ImageCollection(VIIRS)
       .filterBounds(region)
       .filterDate(START, END)
       .map(prep_viirs))

nl_med = col.median().rename("rad")

if SMOOTH_RADIUS_M > 0:
    kernel = ee.Kernel.circle(radius=SMOOTH_RADIUS_M, units='meters', normalize=True)
    nl_med = nl_med.focal_mean(kernel=kernel, iterations=1)

# Percentile stretch (server-side)
p = nl_med.reduceRegion(ee.Reducer.percentile([5,95]), region, 500, maxPixels=1e13, bestEffort=True)
vmin = (p.getNumber("rad_p5").getInfo() if p.get("rad_p5") else 0)
vmax = (p.getNumber("rad_p95").getInfo() if p.get("rad_p95") else 40)

vis = {"min": vmin, "max": vmax,
       "palette": ["000004","1f0c48","550f6d","88226a","b63679","e65164","fb8761","fec287","fcfdbf"]}

# --- quick preview (ipyleaflet) ---
m = geemap.Map(center=[31.5204, 74.3587], zoom=10)
m.addLayer(nl_med, vis, "VIIRS NL Oct 2024 (rad)")
m.addLayer(ucs_fc.style(color="white", fillColor="00000000", width=1), {}, "UC boundaries")
m
m.save("NL_Oct2024_preview.html")

[OK] Using: NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG


Map(center=[31.5204, 74.3587], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

In [6]:
# ---- GeoTIFF export (robust across geemap versions)
def export_image_local(img, filename, region, scale=500, crs="EPSG:4326"):
    import geemap, urllib.request
    try:
        geemap.ee_export_image(img.clip(region), filename, scale=scale, region=region, file_per_band=False)
        print(f"[OK] GeoTIFF → {filename}")
        return
    except TypeError:
        pass
    except Exception as e:
        print("[INFO] ee_export_image failed:", e)
    try:
        geemap.download_ee_image(img.clip(region), filename=filename, scale=scale, region=region, crs=crs)
        print(f"[OK] GeoTIFF (download_ee_image) → {filename}")
        return
    except Exception as e:
        print("[INFO] download_ee_image failed:", e)
    url = img.clip(region).getDownloadURL({"scale": scale, "crs": crs, "region": region, "filePerBand": False})
    urllib.request.urlretrieve(url, filename)
    print(f"[OK] GeoTIFF (getDownloadURL) → {filename}")

tif_path = "VIIRS_NL_Lahore_Oct2024_rad_500m.tif"
export_image_local(nl_med, tif_path, region, scale=500)


Generating URL ...
Please wait ...
Data downloaded to /Users/ahmed/Desktop/Senior Fall 25/SPROJ - Dr Tahir/SPROJ/notebooks/NL/VIIRS_NL_Lahore_Oct2024_rad_500m.tif
[OK] GeoTIFF → VIIRS_NL_Lahore_Oct2024_rad_500m.tif


In [7]:
import geemap, geopandas as gpd, pandas as pd, os

SAMPLE_SCALE = 250               # 250 m points look smooth; keep ≤ 250 to avoid huge files
OUT_PREFIX   = "VIIRS_NL_Lahore_Oct2024"

# 1) EE → points FeatureCollection (lon/lat + radiance 'val')
pts_fc = ee.Image.pixelLonLat().addBands(nl_med.rename("val")).sample(
    region=region, scale=SAMPLE_SCALE, geometries=True, seed=1
)

# 2) GeoJSON (portable for Folium & QGIS)
geojson_pts = f"{OUT_PREFIX}_points_{SAMPLE_SCALE}m.geojson"
geemap.ee_export_vector(pts_fc, filename=geojson_pts)
print(f"[OK] GeoJSON points → {geojson_pts}")

# 3) CSV (fast Folium HeatMap)
try:
    df_pts = geemap.ee_to_df(pts_fc)
except Exception:
    info = ee.FeatureCollection(pts_fc).getInfo()
    df_pts = pd.DataFrame([f["properties"] for f in info["features"]])
df_pts = df_pts.rename(columns={"latitude":"lat","longitude":"lon"})
df_pts[["lat","lon","val"]].to_csv(f"{OUT_PREFIX}_points_{SAMPLE_SCALE}m.csv", index=False)
print(f"[OK] CSV points → {OUT_PREFIX}_points_{SAMPLE_SCALE}m.csv")

# 4) Shapefile (ESRI set: .shp/.shx/.dbf/.prj)
gdf = gpd.read_file(geojson_pts)
if gdf.crs is None or gdf.crs.to_epsg() != 4326:
    gdf = gdf.set_crs(4326, allow_override=True)
shp_dir  = f"{OUT_PREFIX}_points_{SAMPLE_SCALE}m_shp"
os.makedirs(shp_dir, exist_ok=True)
shp_path = os.path.join(shp_dir, f"{OUT_PREFIX}_points_{SAMPLE_SCALE}m.shp")
gdf.to_file(shp_path, driver="ESRI Shapefile")
print(f"[OK] Shapefile set → {shp_dir}  (", ", ".join(sorted(os.listdir(shp_dir))), ")")


Generating URL ...
Please wait ...
Data downloaded to /Users/ahmed/Desktop/Senior Fall 25/SPROJ - Dr Tahir/SPROJ/notebooks/NL/VIIRS_NL_Lahore_Oct2024_points_250m.geojson
[OK] GeoJSON points → VIIRS_NL_Lahore_Oct2024_points_250m.geojson
[OK] CSV points → VIIRS_NL_Lahore_Oct2024_points_250m.csv
[OK] Shapefile set → VIIRS_NL_Lahore_Oct2024_points_250m_shp  ( VIIRS_NL_Lahore_Oct2024_points_250m.cpg, VIIRS_NL_Lahore_Oct2024_points_250m.dbf, VIIRS_NL_Lahore_Oct2024_points_250m.prj, VIIRS_NL_Lahore_Oct2024_points_250m.shp, VIIRS_NL_Lahore_Oct2024_points_250m.shx )


In [8]:


# Optional smoothing for a softer, interpolated look
if SMOOTH_RADIUS_M > 0:
    kernel = ee.Kernel.circle(radius=SMOOTH_RADIUS_M, units='meters', normalize=True)
    nl_med = nl_med.focal_mean(kernel=kernel, iterations=1)

# ---------- 4) Robust visualization stretch (percentiles over region) ----------
p = nl_med.reduceRegion(
    reducer=ee.Reducer.percentile([5, 95]),
    geometry=region, scale=500, maxPixels=1e13, bestEffort=True
)
p5  = p.getNumber("rad_p5")
p95 = p.getNumber("rad_p95")
vmin = (p5.getInfo() if p5 else 0)     # nW/cm²/sr
vmax = (p95.getInfo() if p95 else 40)

vis = {"min": vmin, "max": vmax,
       "palette": ["000004","1f0c48","550f6d","88226a","b63679","e65164","fb8761","fec287","fcfdbf"]}

# ---------- 5) (A) Export GeoTIFF (radiance) ----------
def export_image_local(img, filename, region, scale=500, crs="EPSG:4326"):
    # Try multiple geemap routes to avoid version issues
    try:
        geemap.ee_export_image(img.clip(region), filename, scale=scale, region=region, file_per_band=False)
        print(f"[OK] Exported via geemap.ee_export_image → {filename}")
        return
    except TypeError:
        pass
    except Exception as e:
        print("[INFO] ee_export_image failed:", e)
    try:
        geemap.download_ee_image(img.clip(region), filename=filename, scale=scale, region=region, crs=crs)
        print(f"[OK] Exported via geemap.download_ee_image → {filename}")
        return
    except Exception as e:
        print("[INFO] download_ee_image failed:", e)
    # Last resort: URL
    import urllib.request
    url = img.clip(region).getDownloadURL({
        "scale": scale, "crs": crs, "region": region, "filePerBand": False
    })
    urllib.request.urlretrieve(url, filename)
    print(f"[OK] Exported via getDownloadURL → {filename}")

tif_path = f"{OUT_PREFIX}_rad_500m.tif"
export_image_local(nl_med, tif_path, region, scale=500)
print("[OK] Saved raster →", tif_path)

# ---------- 5) (B) Export point grid (GeoJSON + CSV) for Folium heatmap ----------
pts_fc = ee.Image.pixelLonLat().addBands(nl_med.rename("val")).sample(
    region=region, scale=SAMPLE_SCALE, geometries=True, seed=1
)
geojson_pts = f"{OUT_PREFIX}_points_{SAMPLE_SCALE}m.geojson"
geemap.ee_export_vector(pts_fc, filename=geojson_pts)
print("[OK] Saved GeoJSON points →", geojson_pts)

# CSV for super-fast Folium HeatMap
try:
    df_pts = geemap.ee_to_df(pts_fc)
except Exception:
    info = ee.FeatureCollection(pts_fc).getInfo()
    df_pts = pd.DataFrame([f["properties"] for f in info["features"]])
df_pts = df_pts.rename(columns={"latitude":"lat","longitude":"lon"})
df_pts[["lat","lon","val"]].to_csv(f"{OUT_PREFIX}_points_{SAMPLE_SCALE}m.csv", index=False)
print("[OK] Saved CSV points →", f"{OUT_PREFIX}_points_{SAMPLE_SCALE}m.csv")



KeyboardInterrupt: 

In [ ]:
# ---------- 5) (C) UC-wise averages ----------
uc_stats = nl_med.reduceRegions(
    collection=ucs_fc,
    reducer=ee.Reducer.mean(),
    scale=500,
    tileScale=2
)
try:
    df_uc = geemap.ee_to_df(uc_stats)
except Exception:
    info = ee.FeatureCollection(uc_stats).getInfo()
    df_uc = pd.DataFrame([f["properties"] for f in info["features"]])
df_uc = df_uc.rename(columns={"mean":"rad_mean"})
df_uc.to_csv(f"{OUT_PREFIX}_UC_stats.csv", index=False)
print("[OK] Saved UC stats →", f"{OUT_PREFIX}_UC_stats.csv")

# ---------- 6) Quick interactive preview (optional; opens in notebook) ----------
Map = geemap.Map(center=[31.5204, 74.3587], zoom=10)
Map.addLayer(nl_med, vis, "VIIRS NL (nW/cm²/sr) — Oct 2024")
Map.addLayer(ucs_fc.style(color="white", fillColor="00000000", width=1), {}, "UC boundaries")
Map.to_html(f"{OUT_PREFIX}_map.html")
print("[OK] Saved HTML preview →", f"{OUT_PREFIX}_map.html")